Importing libraries and datasets

In [ ]:
!pip install beautifulsoup4 pandas requests tqdm

import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from urllib.parse import quote_plus
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import time

In [ ]:
# List of all fortune 500 companies that will be used
forbes = pd.read_csv('https://raw.githubusercontent.com/Dguz06/Datasets/refs/heads/main/Data580F/final%20project/Forbes%20Companies%20-%20Sheet1.csv')
companies = forbes["Company"]
companies


,Company
0,Amazon
1,Walmart
2,UnitedHealth
3,Apple
4,Alphabet
...,...
495,Electronic Arts
496,Newell Brands
497,Ingredion
498,KeyCorp


Scraping all headlines for all companies

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0"}


def fetch_day_company(date, company, max_results=10):

    time.sleep(0.052)
    next_day = date + timedelta(days=1)
    encoded_query = quote_plus(company)

    url = (
        "https://news.google.com/rss/search?"
        f"q={encoded_query}+after:{date.strftime('%Y-%m-%d')}+before:{next_day.strftime('%Y-%m-%d')}"
        "&hl=en-US&gl=US&ceid=US:en"
    )

    try:

        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.content, "xml")

        items = soup.find_all("item")

        seen_titles = set()
        data = []

        for item in items:

            title = item.title.text.strip()

            if title in seen_titles:
                continue

            seen_titles.add(title)

            pub_date = datetime.strptime(
                item.pubDate.text,
                "%a, %d %b %Y %H:%M:%S %Z"
            )

            source = item.source.text if item.source else None
            link = item.link.text

            data.append({
                "company": company,
                "date": pub_date.strftime("%Y-%m-%d"),
                "rank": len(data) + 1,
                "title": title,
                "source": source,
                "link": link
            })

            if len(data) >= max_results:
                break

        return data

    except:
        return []



def scraper(companies, start_date, end_date, workers=30):

    tasks = []

    current = start_date
    while current <= end_date:

        for company in companies:
            tasks.append((current, company))

        current += timedelta(days=1)

    results = []

    with ThreadPoolExecutor(max_workers=workers) as executor:

        futures = [
            executor.submit(fetch_day_company, d, c)
            for d, c in tasks
        ]

        for f in tqdm(as_completed(futures), total=len(futures)):
            results.extend(f.result())

    df = pd.DataFrame(results)

    df.to_csv("google_news_dataset.csv", index=False)

    print("Saved dataset")
    print(df.head())

    return df

In [ ]:

df = scraper(
    companies=companies,
    start_date=datetime(2025,1,1),
    end_date=datetime(2025,12,31),
    workers=30
)

100%|██████████| 182500/182500 [1:23:50<00:00, 36.28it/s]


Saved dataset
             company        date  rank  \
0  Johnson & Johnson  2025-01-02     1   
1  Johnson & Johnson  2025-01-02     2   
2  Johnson & Johnson  2025-01-02     3   
3  Johnson & Johnson  2025-01-02     4   
4  Johnson & Johnson  2025-01-02     5   

                                               title                 source  \
0  Johnson & Johnson Declares Dividend, Short Int...          Yahoo Finance   
1  Pharma giant secures land for billion-dollar e...  The Business Journals   
2        Top 10 medtech deals of 2024 - MedTech Dive           MedTech Dive   
3  STAT6 stat sig? Kaken, J&J $1.2B-plus deal add...       BioWorld MedTech   
4  FDA red and green lights: December 2024 | Apex...      Oncology Pipeline   

                                                link  
0  https://news.google.com/rss/articles/CBMijgFBV...  
1  https://news.google.com/rss/articles/CBMirAFBV...  
2  https://news.google.com/rss/articles/CBMicEFVX...  
3  https://news.google.com/rss/article

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/company_articles.csv'

df.to_csv(file_path, index=False)

Mounted at /content/drive


Using finBERT to gather sentiment analysis

In [ ]:
import pandas as pd
data = pd.read_csv("company_articles.csv")

In [ ]:
data

,company,date,rank,title,source,link
0,Johnson & Johnson,2025-01-02,1,"Johnson & Johnson Declares Dividend, Short Int...",Yahoo Finance,https://news.google.com/rss/articles/CBMijgFBV...
1,Johnson & Johnson,2025-01-02,2,Pharma giant secures land for billion-dollar e...,The Business Journals,https://news.google.com/rss/articles/CBMirAFBV...
2,Johnson & Johnson,2025-01-02,3,Top 10 medtech deals of 2024 - MedTech Dive,MedTech Dive,https://news.google.com/rss/articles/CBMicEFVX...
3,Johnson & Johnson,2025-01-02,4,"STAT6 stat sig? Kaken, J&J $1.2B-plus deal add...",BioWorld MedTech,https://news.google.com/rss/articles/CBMipgFBV...
4,Johnson & Johnson,2025-01-02,5,FDA red and green lights: December 2024 | Apex...,Oncology Pipeline,https://news.google.com/rss/articles/CBMihwFBV...
...,...,...,...,...,...,...
1229074,Electronic Arts,2026-01-01,3,List of 27 Acquisitions by Electronic Arts (Ja...,Tracxn,https://news.google.com/rss/articles/CBMisAFBV...
1229075,Electronic Arts,2025-12-31,4,Investment Banker Invasion: How Boutique Firms...,Variety,https://news.google.com/rss/articles/CBMivwFBV...
1229076,Electronic Arts,2025-12-31,5,Fans Accuse Battlefield 6 of Copying Call of D...,Insider Gaming,https://news.google.com/rss/articles/CBMifkFVX...
1229077,Electronic Arts,2026-01-01,6,Region’s wealth funds racked up $119bn of spen...,Arabian Gulf Business Insight | AGBI,https://news.google.com/rss/articles/CBMiogFBV...


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch
import pandas as pd
import os
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model.to(device)
model.eval()

chunk_size = 2000
batch_size = 16
output_dir = "chunks"

os.makedirs(output_dir, exist_ok=True)

num_rows = len(data)

for start in range(0, num_rows, chunk_size):
    end = start + chunk_size
    chunk_file = f"{output_dir}/chunk_{start}.csv"

    if os.path.exists(chunk_file):
        print(f"Skipping {start}-{end} (already done)")
        continue

    print(f"Processing rows {start} to {end}")

    chunk = data.iloc[start:end].copy()
    texts = chunk["title"].fillna("").astype(str).tolist()

    all_preds = []

    with torch.no_grad():
      for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.cuda.amp.autocast():
            outputs = model(**inputs)

        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        all_preds.append(probs.cpu())

    preds = torch.cat(all_preds).numpy()

    chunk["negative"] = preds[:, 0]
    chunk["neutral"]  = preds[:, 1]
    chunk["positive"] = preds[:, 2]

    chunk.to_csv(chunk_file, index=False)


cuda
Processing rows 0 to 2000


/tmp/ipykernel_752/1177587238.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Processing rows 2000 to 4000


/tmp/ipykernel_752/1177587238.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Processing rows 4000 to 6000


...

/tmp/ipykernel_752/1177587238.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Processing rows 1226000 to 1228000


/tmp/ipykernel_752/1177587238.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Processing rows 1228000 to 1230000


/tmp/ipykernel_752/1177587238.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("chunks/chunk_*.csv"))
final_df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/sentiment.csv'

final_df.to_csv(file_path, index=False)

Mounted at /content/drive
